# Liver Disease Prediction System
Full step-by-step analysis: Data Understanding -> EDA -> Cleaning -> Preprocessing -> Modeling -> Evaluation -> Final Model.

> Educational project — not a medical diagnostic tool.

## Phase 1 — Load & Understand the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/liver_dataset.csv", encoding="latin1")
df.columns = [c.strip() for c in df.columns]
df.head()


In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.tail()

### Target column: `Result`
1 = liver disease, 2 = no liver disease (original encoding).

In [ ]:
print(df['Result'].value_counts())
print(df['Result'].value_counts(normalize=True) * 100)


### Data Quality Checks

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct})


In [ ]:
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")


In [ ]:
df['Gender of the patient'].unique()

## Phase 2 — Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
df['Gender of the patient'].value_counts(dropna=False).plot(kind='bar', title='Gender distribution')
plt.show()


In [ ]:
numerical_cols = [c for c in df.columns if c not in ['Gender of the patient', 'Result']]
df[numerical_cols].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flatten(), numerical_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[numerical_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flatten(), numerical_cols):
    sns.boxplot(x='Result', y=col, data=df, ax=ax)
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()


## Phase 3 — Data Cleaning

In [ ]:
df['Result'] = df['Result'].map({1: 1, 2: 0})
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)


In [ ]:
df = df.dropna(subset=['Gender of the patient', 'Age of the patient']).reset_index(drop=True)
print(df.shape)


## Phase 4 — Prepare Data for ML

In [ ]:
X = df.drop(columns=['Result'])
y = df['Result']

gender_map = {"Male": 1, "Female": 0}
X['Gender of the patient'] = X['Gender of the patient'].map(gender_map)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape


## Phase 5 — Missing Values & Outliers (train-only fitting)

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_train[numerical_cols] = imputer.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = imputer.transform(X_test[numerical_cols])


In [ ]:
outlier_summary = {}
for col in numerical_cols:
    Q1, Q3 = X_train[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outlier_summary[col] = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
pd.Series(outlier_summary)


## Phase 6 — Class Imbalance

In [ ]:
y_train.value_counts(normalize=True)

Handled via `class_weight='balanced'` in the models rather than SMOTE/oversampling, to keep the pipeline simple.

## Phase 7 — Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])


## Phase 8 — Feature Engineering
Not performed — the original features were already sufficiently informative based on the EDA above.

## Phase 9 & 10 — Model Training & Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42, max_depth=8),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=42, n_estimators=200),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }
    print(f"--- {name} ---")
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))


## Phase 11 — Model Comparison

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df


## Phase 12 — Hyperparameter Tuning (Random Forest)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [8, 12, None],
    "min_samples_leaf": [1, 3],
}
grid_search = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid, scoring="recall", cv=3, n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)
grid_search.best_params_, grid_search.best_score_


## Phase 13 — Final Model Evaluation

In [ ]:
final_model = grid_search.best_estimator_
final_preds = final_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, final_preds))
print("Precision:", precision_score(y_test, final_preds))
print("Recall:", recall_score(y_test, final_preds))
print("F1:", f1_score(y_test, final_preds))
print(confusion_matrix(y_test, final_preds))
print(classification_report(y_test, final_preds))


## Phase 14 — Save Final Model & Preprocessing Artifacts

In [ ]:
import joblib

joblib.dump(final_model, "../models/model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(imputer, "../models/imputer.pkl")
joblib.dump(gender_map, "../models/encoder.pkl")
joblib.dump(numerical_cols, "../models/numerical_columns.pkl")
joblib.dump(list(X.columns), "../models/feature_order.pkl")
print("Artifacts saved.")


## Summary

See `README.md` in the project root for the full write-up: dataset issues found, preprocessing decisions, model comparison table, why Random Forest was chosen, the caveat about this dataset's duplication inflating accuracy, and project limitations.